# Comparación TinyFaceDetector vs FaceMesh→recorte (multi-sesión)

Proyecto AURA BUAP × IRIS UDLAP. Consolida **todos** los CSV exportados por
`experimentos/alternar-edad-facemesh.html` — barrido de margen y condiciones
difíciles (ángulo lateral, distancia, iluminación, accesorios) — en una sola
tabla comparativa, en vez de leerlos uno por uno a mano.

**Importante — esto NO mide precisión de edad.** Aquí no hay ground truth (es
video en vivo, no un dataset etiquetado como UTKFace). Lo que se mide es:
- **Tasa de detección** por pipeline y condición (esto sí es objetivo)
- **Acuerdo entre pipelines** (qué tan cerca están sus medianas entre sí) —
  no "quién tiene razón", solo qué tanto se separan bajo cada condición

```bash
source sentivision/bin/activate
pip install pandas numpy matplotlib
jupyter notebook
```


## 1. Cargar todas las sesiones

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

pd.set_option("display.precision", 2)
plt.rcParams["figure.dpi"] = 110

archivos = sorted(glob.glob("comparacion_edad_facemesh_*.csv"))
if not archivos:
    archivos = sorted(glob.glob("**/comparacion_edad_facemesh_*.csv", recursive=True))
assert archivos, "No se encontró ningún comparacion_edad_facemesh_*.csv en esta carpeta ni subcarpetas"

print(f"{len(archivos)} sesiones encontradas:")
for a in archivos:
    print(" -", a)


## 2. Resumen por sesión

Una fila por archivo CSV (= una sesión de prueba). El margen y la condición
se toman como el valor más frecuente dentro de esa sesión (por si moviste el
slider a medio camino).


In [ ]:
def moda(serie):
    vals = serie.dropna()
    if len(vals) == 0:
        return None
    return Counter(vals).most_common(1)[0][0]

filas = []
for path in archivos:
    df = pd.read_csv(path)
    n = len(df)

    tfd_det = df["tfd_edad_instantanea"].notna()
    mesh_det = df["mesh_edad_instantanea"].notna()

    condicion = moda(df["condicion"]) if "condicion" in df.columns else "sin_etiquetar"
    margen = moda(df["margen_recorte_pct"]) if "margen_recorte_pct" in df.columns else None

    tfd_mediana = df.loc[tfd_det, "tfd_edad_instantanea"].median()
    mesh_mediana = df.loc[mesh_det, "mesh_edad_instantanea"].median()

    filas.append({
        "archivo": path.split("/")[-1],
        "condicion": condicion,
        "margen_pct": margen,
        "n_muestras": n,
        "tfd_tasa_deteccion": round(tfd_det.mean() * 100, 1),
        "mesh_tasa_deteccion": round(mesh_det.mean() * 100, 1),
        "gap_deteccion_mesh_menos_tfd": round((mesh_det.mean() - tfd_det.mean()) * 100, 1),
        "tfd_mediana_sesion": tfd_mediana,
        "mesh_mediana_sesion": mesh_mediana,
        "diferencia_medianas": round(abs((mesh_mediana or 0) - (tfd_mediana or 0)), 1)
            if pd.notna(mesh_mediana) and pd.notna(tfd_mediana) else None,
    })

resumen = pd.DataFrame(filas)
resumen


## 3. ¿Bajo qué condición se separan más los dos pipelines?

Ordenado de mayor a menor brecha de detección (`mesh - tfd`). Si la hipótesis
del reporte de diagnóstico es correcta (TinyFaceDetector se degrada con
ángulos, distancia y accesorios), las condiciones difíciles deberían tener
gap_deteccion más grande que "normal".


In [ ]:
resumen.sort_values("gap_deteccion_mesh_menos_tfd", ascending=False)[
    ["archivo", "condicion", "margen_pct", "tfd_tasa_deteccion", "mesh_tasa_deteccion", "gap_deteccion_mesh_menos_tfd"]
]


In [ ]:
por_condicion = resumen.groupby("condicion", observed=True).agg(
    n_sesiones=("archivo", "size"),
    tfd_tasa_prom=("tfd_tasa_deteccion", "mean"),
    mesh_tasa_prom=("mesh_tasa_deteccion", "mean"),
    gap_prom=("gap_deteccion_mesh_menos_tfd", "mean"),
).round(1).sort_values("gap_prom", ascending=False)
por_condicion


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(por_condicion))
width = 0.38
ax.bar(x - width/2, por_condicion["tfd_tasa_prom"], width, label="TinyFaceDetector", color="#2a78d6")
ax.bar(x + width/2, por_condicion["mesh_tasa_prom"], width, label="FaceMesh → recorte", color="#4ade80")
ax.axhline(92, color="gray", linestyle="--", linewidth=1, label="Umbral Recall>0.92 (Sección 5.1)")
ax.set_xticks(x)
ax.set_xticklabels(por_condicion.index, rotation=20, ha="right")
ax.set_ylabel("Tasa de detección promedio (%)")
ax.set_title("Tasa de detección por condición")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Efecto del margen de recorte

Agrupa **por valor de margen**, no por archivo — así sirve tanto si hiciste
una sesión separada por margen como si barriste el slider en vivo dentro de
una sola grabación continua (0→25→50→80→100%, ~10s cada uno). Cada fila del
CSV ya trae su propio `margen_recorte_pct`, así que no importa cómo lo hayas
grabado.


In [ ]:
# Junta TODAS las sesiones (puede ser una sola grabación continua con el
# slider en movimiento, o varias sesiones separadas) y agrupa por margen real
# de cada fila, no por archivo completo.
todas = pd.concat([pd.read_csv(a).assign(archivo=a.split("/")[-1]) for a in archivos], ignore_index=True)

por_margen = todas.groupby("margen_recorte_pct").agg(
    n=("mesh_edad_instantanea", "size"),
    tfd_tasa_deteccion=("tfd_edad_instantanea", lambda s: round(s.notna().mean() * 100, 1)),
    mesh_tasa_deteccion=("mesh_edad_instantanea", lambda s: round(s.notna().mean() * 100, 1)),
    tfd_mediana=("tfd_edad_instantanea", "median"),
    mesh_mediana=("mesh_edad_instantanea", "median"),
).reset_index().sort_values("margen_recorte_pct")

por_margen["diferencia_medianas"] = (por_margen["mesh_mediana"] - por_margen["tfd_mediana"]).abs().round(1)

if len(por_margen) < 2:
    print("Solo se encontró un valor de margen en los datos — no hay barrido que graficar todavía.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(por_margen["margen_recorte_pct"], por_margen["diferencia_medianas"], marker="o", color="#e34948")
    axes[0].set_xlabel("Margen de recorte (%)")
    axes[0].set_ylabel("Diferencia de medianas (años)")
    axes[0].set_title("¿Acerca el margen a FaceMesh a TinyFaceDetector?")
    axes[0].grid(alpha=0.3)

    axes[1].plot(por_margen["margen_recorte_pct"], por_margen["mesh_tasa_deteccion"], marker="o", label="FaceMesh", color="#4ade80")
    axes[1].plot(por_margen["margen_recorte_pct"], por_margen["tfd_tasa_deteccion"], marker="o", label="TinyFaceDetector", color="#2a78d6")
    axes[1].set_xlabel("Margen de recorte (%)")
    axes[1].set_ylabel("Tasa de detección (%)")
    axes[1].set_title("Tasa de detección por margen")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

por_margen


## 5. Conclusión (completar tras correr)

- Condición con mayor brecha de detección a favor de FaceMesh: ___
- ¿Alguna condición donde TinyFaceDetector iguala o supera a FaceMesh? ___
- Margen recomendado (el que minimiza diferencia de medianas sin agregar
  demasiado fondo/contexto): ___%
- **Decisión sobre integrar a `app.js`:** ___
